# 01（US）拉取市场数据（Stooq：交易日历 + 宽基指数日线）

目的：拉取美国宽基指数日线数据（Stooq，免 API key），并缓存到 `data/cache_us/`，供后续“华盛顿时间口径：立春年/天干地支 → 美股涨跌”研究使用。

输出：
- `data/cache_us/index_daily/{symbol_id}.csv.gz`
- `data/cache_us/trade_cal/trade_cal.csv.gz`

说明：
- 本 notebook 需要联网下载 CSV；若某个指数下载失败会跳过，其它指数仍可继续。
- 交易日历 `trade_cal` 由“成功下载的指数数据”的 `trade_date` union 构造。


In [1]:
from __future__ import annotations

import datetime as _dt
import re
from pathlib import Path
from typing import Iterable
from urllib.parse import quote
from urllib.request import Request, urlopen

import pandas as pd
from zoneinfo import ZoneInfo


def find_project_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()

    # Typical: notebook is executed under project root or under ./notebooks / ./notebooks_US
    for candidate in [here] + list(here.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    # Fallback: if executed from a parent folder, look for a child project dir
    for candidate in here.glob('*'):
        if candidate.is_dir() and (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    return here


ROOT = find_project_root()
print('PROJECT_ROOT =', ROOT)

# =====================
# Config (edit as needed)
# =====================
START_DATE = '20100101'
END_DATE = _dt.datetime.now(tz=ZoneInfo('America/New_York')).strftime('%Y%m%d')
FORCE_REFRESH = False  # True: ignore local cache

# Stooq symbols (indices typically start with '^')
INDEX_LIST = {
    'SPX': '^spx',  # S&P 500
    'NDQ': '^ndq',  # Nasdaq Composite
    'NDX': '^ndx',  # Nasdaq 100
    'DJI': '^dji',  # Dow Jones Industrial Average
}

CACHE_DIR = ROOT / 'data/cache_us'
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def sanitize_symbol_id(symbol: str) -> str:
    s = symbol.strip().lower()
    s = re.sub(r'[^0-9a-z]+', '_', s)
    s = s.strip('_')
    return s or 'symbol'


def read_csv_gz(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, compression='gzip', dtype={'trade_date': str})


def write_csv_gz(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, compression='gzip')


def fetch_stooq_daily(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    # Stooq daily CSV endpoint
    url = f"https://stooq.com/q/d/l/?s={quote(symbol)}&d1={start_date}&d2={end_date}&i=d"
    req = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urlopen(req, timeout=30) as resp:
        raw = resp.read().decode('utf-8', errors='replace')

    # Stooq returns CSV with header like: Date,Open,High,Low,Close,Volume
    from io import StringIO

    df = pd.read_csv(StringIO(raw))
    if df.empty:
        return df

    df.columns = [c.strip().lower() for c in df.columns]
    if 'date' not in df.columns or 'close' not in df.columns:
        raise ValueError(f'Unexpected Stooq columns for {symbol}: {df.columns.tolist()}')

    out = df.copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce')
    out = out.dropna(subset=['date']).copy()
    out['trade_date'] = out['date'].dt.strftime('%Y%m%d')

    keep = ['trade_date']
    for col in ['open', 'high', 'low', 'close', 'volume']:
        if col in out.columns:
            keep.append(col)

    out = out[keep].copy()
    out = out.sort_values(['trade_date']).drop_duplicates(subset=['trade_date']).reset_index(drop=True)
    return out


print('END_DATE =', END_DATE)

idx_dir = CACHE_DIR / 'index_daily'
idx_dir.mkdir(parents=True, exist_ok=True)

summary = []
open_days: set[str] = set()

for name, symbol in INDEX_LIST.items():
    symbol_id = sanitize_symbol_id(symbol)
    out_path = idx_dir / f'{symbol_id}.csv.gz'

    if out_path.exists() and not FORCE_REFRESH:
        df = read_csv_gz(out_path)
        print(f'[{name}] Loaded cache:', out_path)
    else:
        try:
            df = fetch_stooq_daily(symbol, START_DATE, END_DATE)
        except Exception as exc:
            print(f'[{name}] Fetch failed for {symbol}: {exc}')
            continue

        if df.empty:
            print(f'[{name}] Empty result for {symbol} (skip)')
            continue

        df.insert(0, 'symbol', symbol)
        df.insert(1, 'symbol_id', symbol_id)
        write_csv_gz(df, out_path)
        print(f'[{name}] Saved cache:', out_path)

    if 'trade_date' in df.columns:
        open_days.update(df['trade_date'].astype(str).tolist())

    summary.append(
        {
            'name': name,
            'symbol': symbol,
            'symbol_id': symbol_id,
            'rows': int(len(df)),
            'min_trade_date': df['trade_date'].min() if len(df) else None,
            'max_trade_date': df['trade_date'].max() if len(df) else None,
        }
    )

summary_df = pd.DataFrame(summary).sort_values(['name']).reset_index(drop=True)
display(summary_df)

# 交易日历：用成功下载的 trade_date union 构造
if not open_days:
    raise RuntimeError('No index data downloaded. Please check network access to stooq.com and symbols in INDEX_LIST.')

cal_dates = sorted(open_days)
trade_cal = pd.DataFrame({'cal_date': cal_dates})
trade_cal['exchange'] = 'US'
trade_cal['is_open'] = 1
trade_cal['pretrade_date'] = trade_cal['cal_date'].shift(1)

trade_cal_path = CACHE_DIR / 'trade_cal' / 'trade_cal.csv.gz'
write_csv_gz(trade_cal, trade_cal_path)
print('Saved cache:', trade_cal_path)

print('trade_cal open days =', len(trade_cal))
print('range =', trade_cal['cal_date'].min(), '->', trade_cal['cal_date'].max())
trade_cal.head()


PROJECT_ROOT = D:\Work\中国传统投资\风水五行阴阳天干地支
END_DATE = 20260215
[SPX] Saved cache: D:\Work\中国传统投资\风水五行阴阳天干地支\data\cache_us\index_daily\spx.csv.gz
[NDQ] Saved cache: D:\Work\中国传统投资\风水五行阴阳天干地支\data\cache_us\index_daily\ndq.csv.gz
[NDX] Saved cache: D:\Work\中国传统投资\风水五行阴阳天干地支\data\cache_us\index_daily\ndx.csv.gz
[DJI] Saved cache: D:\Work\中国传统投资\风水五行阴阳天干地支\data\cache_us\index_daily\dji.csv.gz


,name,symbol,symbol_id,rows,min_trade_date,max_trade_date
0,DJI,^dji,dji,4054,20100104,20260213
1,NDQ,^ndq,ndq,4055,20100104,20260213
2,NDX,^ndx,ndx,4054,20100104,20260213
3,SPX,^spx,spx,4054,20100104,20260213


Saved cache: D:\Work\中国传统投资\风水五行阴阳天干地支\data\cache_us\trade_cal\trade_cal.csv.gz
trade_cal open days = 4055
range = 20100104 -> 20260213


,cal_date,exchange,is_open,pretrade_date
0,20100104,US,1,None
1,20100105,US,1,20100104
2,20100106,US,1,20100105
3,20100107,US,1,20100106
4,20100108,US,1,20100107
